In [ ]:
import numpy as np
import pandas as pd

# Set the random seed for reproducibility
np.random.seed(639)

# Define model parameters
n_obs = 100
sigma_x = 2
sigma_e = 1
lag_d = 3

# Simulate predictor and noise series
X_t = np.random.normal(loc=0.0, scale=sigma_x, size=n_obs)
noise = np.random.normal(loc=0.0, scale=sigma_e, size=n_obs)

# Construct the response series with the specified lag
Y_t = np.full(n_obs, np.nan)
Y_t[lag_d:] = X_t[:-lag_d] + noise[lag_d:]

# Combine into a DataFrame for convenient inspection
simulated_data = pd.DataFrame({"time": np.arange(1, n_obs + 1), "X_t": X_t, "Y_t": Y_t})

# Display the first ten observations
simulated_data.head(10)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Prepare the series as pandas objects for alignment
x_series = pd.Series(X_t)
y_series = pd.Series(Y_t)

# Compute the sample cross-correlation across symmetric lags
max_lag = 20
lags = np.arange(-max_lag, max_lag + 1)
ccf_values = []

for lag in lags:
    aligned = pd.concat([x_series.shift(-lag), y_series], axis=1).dropna()
    correlation = aligned.corr().iloc[0, 1]
    ccf_values.append(correlation)

ccf_results = pd.DataFrame({"lag": lags, "ccf": ccf_values})

# Plot the cross-correlation function
sns.set_theme(style="whitegrid")
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(ccf_results["lag"], ccf_results["ccf"], color="#4C72B0", width=0.8)
ax.axhline(0, color="black", linewidth=1.0)

# Add approximate 95% confidence bounds
effective_n = np.count_nonzero(~np.isnan(Y_t[lag_d:]))
bound = 1.96 / np.sqrt(effective_n)
ax.axhline(bound, color="red", linestyle="--", linewidth=1.0)
ax.axhline(-bound, color="red", linestyle="--", linewidth=1.0)

ax.set_xlabel("Lag")
ax.set_ylabel("Sample CCF")
ax.set_title("Sample Cross-Correlation Function between X_t and Y_t")
plt.tight_layout()
plt.savefig("cross_correlation.png", dpi=300, bbox_inches="tight")
plt.show()

ccf_results


In [ ]:
interpretation = (
    "The cross-correlation function displays its strongest positive correlation at lag -3, "
    "indicating that the predictor series X_t leads the response Y_t by three periods. "
    "The positive sign of the peak aligns with the positive coefficient beta_1 = 1 in the model, "
    "showing that higher values of X_{t-3} tend to be associated with higher Y_t. "
    "The shift by -d = -3 manifests as the location of the dominant peak in the CCF, "
    "because the model links Y_t to a three-period lagged value of X_t."
)
print(interpretation)
